# Dua embedding BGE-M3 V2 — Google Colab

**PREPARED, NOT EXECUTED.** Pilih Runtime → Change runtime type → GPU. Buka notebook ini dan jalankan cells berurutan. Python3.12/Linux wajib; GPU/driver harus mendukung torch2.12.1+cu126. Jika error, STOP; tidak ada fallback model, CPU, perubahan precision, atau retry otomatis.

Upload hanya `input_package.zip` (dua canonical serialized inputs + checksum-bound runtime). Tidak perlu upload corpus, qrels, indeks V1, repository, atau credentials. Download model hanya terjadi di Colab saat Anda mengeksekusi cell inference.

Vektor lokal369 dipertahankan; dua output GPU cloud dapat berbeda floating point. Tidak ada klaim bitwise equivalence atau benchmark retrieval. Paket output belum membuat R3 V2 COMPLETE sampai import lokal lolos.

In [ ]:
import sys, subprocess, os, venv
from pathlib import Path
assert sys.version_info[:2] == (3,12), 'Gunakan runtime Python3.12; jangan mengganti package/model secara acak'
subprocess.run(['nvidia-smi'], check=True)
WORK=Path('/content/wo_bge_m3')
WORK.mkdir(exist_ok=False)  # second run stops; no implicit retry/overwrite
os.environ.update(HF_HOME=str(WORK/'hf_cache'), PIP_CACHE_DIR=str(WORK/'pip_cache'), TMP=str(WORK/'tmp'), TEMP=str(WORK/'tmp'))
(WORK/'tmp').mkdir()
venv.EnvBuilder(with_pip=True).create(WORK/'venv')
PY=str(WORK/'venv/bin/python')


## Environment terpisah

Install hanya pada venv Colab. Torch dipin dari official CUDA12.6 index. Dependency core mengikuti versi lokal; transitive resolution dicatat melalui full environment lock dalam output. Dry-run dan pip check wajib PASS; jangan gunakan --no-deps atau mengubah versi jika conflict. Cell ini belum diuji pada real Colab.

In [ ]:
import json
PINS={'FlagEmbedding': '1.4.2', 'accelerate': '1.15.0', 'datasets': '5.0.1', 'fsspec': '2026.6.0', 'huggingface-hub': '0.36.2', 'ir-datasets': '0.6.3', 'numpy': '2.5.3', 'peft': '0.18.1', 'protobuf': '7.36.2', 'psutil': '7.2.2', 'safetensors': '0.8.0', 'sentence-transformers': '5.2.0', 'sentencepiece': '0.2.2', 'tokenizers': '0.22.2', 'torch': '2.12.1+cu126', 'transformers': '4.57.6'}
REQS=['FlagEmbedding==1.4.2', 'accelerate==1.15.0', 'datasets==5.0.1', 'fsspec==2026.6.0', 'huggingface-hub==0.36.2', 'ir-datasets==0.6.3', 'numpy==2.5.3', 'peft==0.18.1', 'protobuf==7.36.2', 'psutil==7.2.2', 'safetensors==0.8.0', 'sentence-transformers==5.2.0', 'sentencepiece==0.2.2', 'tokenizers==0.22.2', 'transformers==4.57.6']
subprocess.run([PY,'-m','pip','install','torch==2.12.1','--index-url','https://download.pytorch.org/whl/cu126'],check=True)
constraint=WORK/'torch_constraint.txt'
constraint.write_text('torch==2.12.1+cu126\n')
plan=WORK/'dependency_plan.json'
subprocess.run([PY,'-m','pip','install','--dry-run','--report',str(plan),'-c',str(constraint),*REQS],check=True)
resolved=json.loads(plan.read_text())
assert not any(x['metadata']['name'].lower()=='torch' for x in resolved['install']), 'Resolver mencoba mengganti torch; STOP'
subprocess.run([PY,'-m','pip','install','-c',str(constraint),*REQS],check=True)
subprocess.run([PY,'-m','pip','check'],check=True)
subprocess.run([PY,'-c',"import torch; assert torch.__version__=='2.12.1+cu126'; assert torch.cuda.is_available(); print(torch.__version__,torch.version.cuda,torch.cuda.get_device_name(0),torch.cuda.get_arch_list())"],check=True)


## Upload dan verifikasi input

Pilih `experiments/portable/v2_bge_m3/input_package.zip` dari laptop. Exact ZIP dan runtime SHA diperiksa sebelum runtime.py dijalankan.

In [ ]:
EXPECTED_ZIP_SHA256='4aaf7b5f122087789a4ec76467a40a6be5f31f57813cc4cfc4db1d4029adf183'
EXPECTED_INPUT_FINGERPRINT='50e54d5d26979ef000a3b41e57c39fef91a91ea8a286488d865da7b5b450477a'
from google.colab import files
import hashlib, zipfile, io
uploaded=files.upload()
assert set(uploaded)=={'input_package.zip'}, 'Upload tepat satu input_package.zip'
raw=uploaded['input_package.zip']
assert hashlib.sha256(raw).hexdigest()==EXPECTED_ZIP_SHA256, 'Input ZIP checksum mismatch'
INPUT=WORK/'input_package.zip'
INPUT.write_bytes(raw)
with zipfile.ZipFile(io.BytesIO(raw)) as archive:
    members=archive.infolist()
    assert len(members)==2 and {x.filename for x in members}=={'inputs.json','runtime.py'}
    assert all(x.file_size <= 2*1024**2 for x in members)
    package=json.loads(archive.read('inputs.json'))
    runtime=archive.read('runtime.py')
assert package['package_fingerprint']==EXPECTED_INPUT_FINGERPRINT
assert hashlib.sha256(runtime).hexdigest()==package['runtime_sha256']
(WORK/'runtime.py').write_bytes(runtime)
print([(r['chunk_id'],r['actual_tokens'],r['input_sha256']) for r in package['inputs']])


## Satu run: checkpoint resmi, dua dense vectors

Snapshot download memakai exact revision; semua model/tokenizer dan official implementation hashes diperiksa. Tokenizer actual diperiksa sebelum weights. Forward mengikuti jalur lokal: local BGEM3FlagModel → `.model` dense-only → float32 → project L2. Batch1, FP16, max_length640 tanpa truncation. Tidak menggunakan wrapper encode yang dapat mencoba batch fallback. Output dibuat hanya setelah dua vectors PASS. OOM/error menghentikan subprocess; jangan rerun otomatis.

In [ ]:
OUTPUT=WORK/'colab_embeddings.zip'
subprocess.run([PY,str(WORK/'runtime.py'),'colab','--input',str(INPUT),'--output',str(OUTPUT)],check=True)
assert OUTPUT.is_file(), 'Tidak ada COMPLETE output'
with zipfile.ZipFile(OUTPUT) as archive:
    result=json.loads(archive.read('manifest.json'))
assert result['completion_status']=='COMPLETE'
print(result['rows'])
print(result['encoding_environment'])
print(result['timings'])
print('Exchange ZIP SHA256:',hashlib.sha256(OUTPUT.read_bytes()).hexdigest())


## Download dan import lokal

Download `colab_embeddings.zip`, tempatkan di `D:/ModelCache/wo-retrieval-benchmark/colab_exchange/v2/`. Minta Codex memverifikasi output sebelum import. Import CLI tidak memuat BGE-M3 atau menjalankan GPU. Manifest COMPLETE di ZIP hanya menyatakan dua-output exchange lengkap, bukan R3 V2 sudah lengkap.

```powershell
$env:PYTHONPATH='src'
$env:PYTHONUTF8='1'
$env:PYTHONIOENCODING='utf-8'
.venv-ml/Scripts/python.exe -B -m wo_retrieval.portable_embeddings import --input experiments/portable/v2_bge_m3/input_package.zip --output D:/ModelCache/wo-retrieval-benchmark/colab_exchange/v2/colab_embeddings.zip
```

Command tersebut menjalankan actual import dan publikasi371vectors hanya jika semua validasi nyata PASS. Jangan jalankan pada milestone preparation ini. Tidak mengaktifkan R4 atau benchmark.

In [ ]:
files.download(str(OUTPUT))
